# O "programa dos sonhos" da FidelidadeScript

Antes de definir o analisador ou as demais fases do compilador, descrevemos o programa que queremos permitir que uma pessoa escreva. O exemplo combina uma compra, o cálculo do cashback, a concessão de pontos e operações comuns de uma conta de fidelidade.

Este programa é o contrato inicial da linguagem: as próximas etapas deverão torná-lo executável e validar as regras de negócio do domínio.

A primeira célula executável valida a ficha antes de definir o programa. Se os requisitos mínimos não forem atendidos, ela lança um erro.

In [ ]:
# ============================================================
# FICHA DE QUALIDADE — validar antes de definir o programa DSL
# ============================================================

import re

MINHA_FICHA = {
    "nome": "FidelidadeScript",
    "dominio": "programas de pontos e cashback",
    "tipos": ["inteiro", "decimal", "texto", "logico"],
    "comandos_dominio": [
        'creditar "<cliente>", <pontos>;',
        'resgatar "<cupom>", <pontos>;',
        'expirar "<cliente>", <pontos>;',
        'extrato "<cliente>";',
    ],
    "regras_de_negocio": [
        "percentual de cashback acima de 100% gera erro",
        "pontos negativos geram erro",
        "resgate acima do saldo disponível gera erro de execução",
    ],
    "programas_exemplo": [
        r'''
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;
se compra >= 200.0 {
    pontos = pontos + 50;
}
creditar "cliente_42", pontos;
extrato "cliente_42";
''',
        r'''
decimal percentual_cashback = 120.0;
inteiro pontos = -5;
se percentual_cashback > 100.0 {
    creditar "cliente_42", pontos;
}
''',
        r'''
inteiro pontos = 50;
creditar "cliente_42", pontos;
se pontos < 100 {
    resgatar "cupom_10", 100;
}
''',
    ],
}

def conferir_ficha(ficha):
    if not isinstance(ficha, dict):
        raise TypeError("A ficha da linguagem deve ser um dicionário.")

    def tem_texto(valor):
        return isinstance(valor, str) and bool(valor.strip())

    def quantidade_itens(campo):
        itens = ficha.get(campo)
        if not isinstance(itens, list):
            return 0
        return sum(tem_texto(item) for item in itens)

    nome = ficha.get("nome")
    dominio = ficha.get("dominio")
    identidade_valida = (
        tem_texto(nome)
        and tem_texto(dominio)
        and "todo" not in (nome + dominio).lower()
    )
    programas = ficha.get("programas_exemplo")
    tres_programas_validos = (
        isinstance(programas, list)
        and len(programas) == 3
        and all(tem_texto(programa) for programa in programas)
    )
    todos_usam_controle = (
        tres_programas_validos
        and all(
            re.search(r"(?m)^\s*(?:se|enquanto)\b", programa)
            for programa in programas
        )
    )
    checagens = [
        ("nome e domínio preenchidos", identidade_valida),
        ("pelo menos 3 tipos", quantidade_itens("tipos") >= 3),
        ("pelo menos 2 comandos do domínio", quantidade_itens("comandos_dominio") >= 2),
        ("pelo menos 2 regras de negócio", quantidade_itens("regras_de_negocio") >= 2),
        ("exatamente 3 programas-exemplo", tres_programas_validos),
        ("cada programa usa se ou enquanto", todos_usam_controle),
    ]

    for texto, ok in checagens:
        print(("✅" if ok else "⬜"), texto)

    faltam = [texto for texto, ok in checagens if not ok]
    if faltam:
        raise ValueError("Ficha da linguagem incompleta: " + "; ".join(faltam))

    print("\n🚀 Ficha completa! O programa de exemplo pode ser carregado.")


conferir_ficha(MINHA_FICHA)

# ============================================================
# PASSO 0 — O "programa dos sonhos" da FidelidadeScript
# ============================================================

PROGRAMA_FIDELIDADE = r'''
// FidelidadeScript — campanha de cashback e pontos
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;

se compra >= 200.0 {
    pontos = pontos + 50;
}

mostrar "Cashback calculado:";
mostrar cashback;
creditar "cliente_42", pontos;
extrato "cliente_42";
resgatar "cupom_10", 100;
expirar "cliente_42", 25;
extrato "cliente_42";
'''

print("Programa com", PROGRAMA_FIDELIDADE.count("\n") - 1, "linhas carregado em PROGRAMA_FIDELIDADE.")

## Passo 1 — Análise léxica

Nesta etapa, o Lark reconhece os tipos, as palavras reservadas e os símbolos da FidelidadeScript. Comentários e espaços são ignorados; erros léxicos informam localização e dica.

In [ ]:
# ============================================================
# PASSO 1 — Análise léxica da FidelidadeScript
# ============================================================

import re
from dataclasses import dataclass

from lark import Lark
from lark.exceptions import UnexpectedCharacters

TERMINAIS = r'''
// ---------- palavras reservadas e tipos ----------
T_INTEIRO: "inteiro"
T_DECIMAL: "decimal"
T_TEXTO: "texto"
T_LOGICO: "logico"
MOSTRAR: "mostrar"
CREDITAR: "creditar"
RESGATAR: "resgatar"
EXPIRAR: "expirar"
EXTRATO: "extrato"
SE: "se"
SENAO: "senao"
ENQUANTO: "enquanto"
E: "e"
OU: "ou"
NAO: "nao"
VERDADEIRO: "verdadeiro"
FALSO: "falso"

// ---------- literais e identificadores ----------
NUM_DECIMAL.2: /[0-9]+\.[0-9]+/
NUM_INTEIRO: /[0-9]+/
TEXTO_LIT: /"[^"\n]*"/
NOME: /(?!(?:inteiro|decimal|texto|logico|mostrar|creditar|resgatar|expirar|extrato|se|senao|enquanto|e|ou|nao|verdadeiro|falso)\b)[a-zA-Z][a-zA-Z0-9_]*/

// ---------- operadores ----------
MAIOR_IGUAL: ">="
MENOR_IGUAL: "<="
IGUAL_IGUAL: "=="
DIFERENTE: "!="
MAIOR: ">"
MENOR: "<"
ATRIB: "="
MAIS: "+"
MENOS: "-"
VEZES: "*"
DIVIDE: "/"

// ---------- pontuação ----------
PONTO_VIRGULA: ";"
VIRGULA: ","
ABRE_CHAVE: "{"
FECHA_CHAVE: "}"
ABRE_PAR: "("
FECHA_PAR: ")"

// ---------- elementos ignorados ----------
COMENTARIO: /\/\/[^\n]*/
%import common.WS
%ignore WS
%ignore COMENTARIO
'''

PALAVRAS_RESERVADAS = {
    "inteiro": "T_INTEIRO",
    "decimal": "T_DECIMAL",
    "texto": "T_TEXTO",
    "logico": "T_LOGICO",
    "mostrar": "MOSTRAR",
    "creditar": "CREDITAR",
    "resgatar": "RESGATAR",
    "expirar": "EXPIRAR",
    "extrato": "EXTRATO",
    "se": "SE",
    "senao": "SENAO",
    "enquanto": "ENQUANTO",
    "e": "E",
    "ou": "OU",
    "nao": "NAO",
    "verdadeiro": "VERDADEIRO",
    "falso": "FALSO",
}

NOMES_TERMINAIS = [
    nome
    for nome in re.findall(r"^([A-Z_]+)(?:\.\d+)?:", TERMINAIS, re.MULTILINE)
    if nome != "COMENTARIO"
]

if len(NOMES_TERMINAIS) < 20:
    raise RuntimeError("A FidelidadeScript precisa definir pelo menos 20 tipos de token.")
if len(PALAVRAS_RESERVADAS) < 8 or not {"creditar", "resgatar", "expirar", "extrato"}.issubset(PALAVRAS_RESERVADAS):
    raise RuntimeError("Faltam palavras reservadas ou comandos do domínio no léxico.")

_lexer_lark = Lark(
    "start: token*\ntoken: " + " | ".join(NOMES_TERMINAIS) + "\n" + TERMINAIS,
    parser="lalr",
    lexer="basic",
)

@dataclass
class Diagnostico:
    fase: str
    mensagem: str
    linha: int
    coluna: int
    dica: str = ""
    nivel: str = "erro"

    def __str__(self):
        local = f"linha {self.linha}, coluna {self.coluna}"
        texto = f"[{self.fase}] {local}: {self.mensagem}"
        prefixo = "AVISO " if self.nivel == "aviso" else ""
        return prefixo + texto + (f"\nDica: {self.dica}" if self.dica else "")

class ErroCompilacao(Exception):
    def __init__(self, diagnosticos):
        self.diagnosticos = diagnosticos
        super().__init__("\n".join(str(item) for item in diagnosticos))

DICAS_LEXICAS = {
    "'": 'Textos usam aspas duplas, por exemplo: "cliente_42".',
    '"': "Literal de texto sem fechamento; feche as aspas na mesma linha.",
    "&": "Para conjunção lógica, use a palavra reservada 'e'.",
    "|": "Para disjunção lógica, use a palavra reservada 'ou'.",
    "!": "Para negação, use 'nao'; para diferença, use '!='.",
    "#": "Comentários começam com //.",
    "%": "Calcule cashback com percentual / 100; '%' não é um operador da linguagem.",
}

def dica_lexica(caractere):
    if caractere in DICAS_LEXICAS:
        return DICAS_LEXICAS[caractere]
    if caractere.isalpha():
        return "Identificadores usam letras sem acento; por exemplo, escreva 'compra'."
    return "Remova o símbolo ou use um operador/pontuação definido pela FidelidadeScript."

def tokenizar(fonte):
    try:
        return list(_lexer_lark.lex(fonte))
    except UnexpectedCharacters as erro:
        caractere = erro.char
        diagnostico = Diagnostico(
            "léxico",
            f"caractere inesperado {caractere!r}",
            erro.line,
            erro.column,
            dica_lexica(caractere),
        )
        raise ErroCompilacao([diagnostico]) from None

def mostrar_tokens(fonte, limite=None):
    tokens = tokenizar(fonte)
    exibidos = tokens if limite is None else tokens[:limite]
    print(f"{'#':>3}  {'TIPO':<16} {'LEXEMA':<24} LINHA:COLUNA")
    print("-" * 62)
    for indice, token in enumerate(exibidos, 1):
        print(f"{indice:>3}  {token.type:<16} {token.value:<24} {token.line}:{token.column}")
    return tokens

print(f"Tipos de token: {len(NOMES_TERMINAIS)} | Palavras reservadas: {len(PALAVRAS_RESERVADAS)}")
print("Tabela de tokens de PROGRAMA_FIDELIDADE:")
TOKENS_PROGRAMA = mostrar_tokens(PROGRAMA_FIDELIDADE)

ERROS_LEXICOS_DEMONSTRACAO = [
    ("aspas simples", "texto cliente = 'cliente_42';"),
    ("operador lógico inválido", "se pontos > 0 & compra > 0 {}"),
    ("identificador com acento", "inteiro prêmio = 10;"),
]

for nome_caso, fonte_errada in ERROS_LEXICOS_DEMONSTRACAO:
    try:
        tokenizar(fonte_errada)
    except ErroCompilacao as erro:
        print(f"\nExemplo de erro ({nome_caso}):\n{erro}")
    else:
        raise AssertionError(f"O exemplo de erro lexical '{nome_caso}' deveria falhar.")

## Passo 2 — Análise sintática e AST

A gramática LALR organiza declarações, comandos e expressões em níveis de precedência. O Transformer converte a árvore sintática em nós `dataclass`; erros indicam a posição e os símbolos esperados.

In [ ]:
# ============================================================
# PASSO 2 — Análise sintática e construção da AST
# ============================================================

from dataclasses import dataclass, fields, is_dataclass
from lark import Lark, Transformer
from lark.exceptions import UnexpectedInput

REGRAS = r'''
start: programa
programa: comando*

?comando: declaracao
        | atribuicao
        | mostrar_stmt
        | se_stmt
        | enquanto_stmt
        | creditar_stmt
        | resgatar_stmt
        | expirar_stmt
        | extrato_stmt

declaracao: tipo NOME (ATRIB expr)? PONTO_VIRGULA
tipo: T_INTEIRO -> tipo_inteiro
    | T_DECIMAL -> tipo_decimal
    | T_TEXTO -> tipo_texto
    | T_LOGICO -> tipo_logico
atribuicao: NOME ATRIB expr PONTO_VIRGULA
mostrar_stmt: MOSTRAR expr PONTO_VIRGULA
se_stmt: SE expr bloco (SENAO bloco)?
enquanto_stmt: ENQUANTO expr bloco
bloco: ABRE_CHAVE comando* FECHA_CHAVE

creditar_stmt: CREDITAR expr VIRGULA expr PONTO_VIRGULA
resgatar_stmt: RESGATAR expr VIRGULA expr PONTO_VIRGULA
expirar_stmt: EXPIRAR expr VIRGULA expr PONTO_VIRGULA
extrato_stmt: EXTRATO expr PONTO_VIRGULA

// Precedência, da menor para a maior: ou, e, comparações, soma, produto e unários.
expr: logica_ou
logica_ou: logica_e (OU logica_e)*
logica_e: logica_nao (E logica_nao)*
logica_nao: NAO logica_nao -> unop
          | comparacao
comparacao: soma_expr ((MAIOR | MENOR | MAIOR_IGUAL | MENOR_IGUAL | IGUAL_IGUAL | DIFERENTE) soma_expr)*
soma_expr: produto_expr ((MAIS | MENOS) produto_expr)*
produto_expr: fator ((VEZES | DIVIDE) fator)*
fator: MENOS fator -> unop
     | MAIS fator -> unop
     | atom
atom: NUM_DECIMAL -> lit_decimal
    | NUM_INTEIRO -> lit_inteiro
    | TEXTO_LIT -> lit_texto
    | VERDADEIRO -> lit_logico
    | FALSO -> lit_logico
    | NOME -> var
    | ABRE_PAR expr FECHA_PAR -> agrupamento
'''

GRAMATICA = REGRAS + TERMINAIS
parser = Lark(GRAMATICA, parser="lalr", lexer="basic")

@dataclass
class Expr:
    linha: int

@dataclass
class Lit(Expr):
    valor: object
    tipo: str = None

@dataclass
class Var(Expr):
    nome: str
    tipo: str = None

@dataclass
class BinOp(Expr):
    op: str
    esq: Expr
    dir: Expr
    tipo: str = None

@dataclass
class UnOp(Expr):
    op: str
    expr: Expr
    tipo: str = None

@dataclass
class Decl:
    linha: int
    tipo: str
    nome: str
    expr: object = None

@dataclass
class Atrib:
    linha: int
    nome: str
    expr: Expr

@dataclass
class Mostrar:
    linha: int
    expr: Expr

@dataclass
class Se:
    linha: int
    cond: Expr
    entao: list
    senao: object = None

@dataclass
class Enquanto:
    linha: int
    cond: Expr
    corpo: list

@dataclass
class Creditar:
    linha: int
    cliente: Expr
    pontos: Expr

@dataclass
class Resgatar:
    linha: int
    cupom: Expr
    pontos: Expr

@dataclass
class Expirar:
    linha: int
    cliente: Expr
    pontos: Expr

@dataclass
class Extrato:
    linha: int
    cliente: Expr

@dataclass
class Programa:
    comandos: list

class MontaAST(Transformer):
    def start(self, filhos):
        return filhos[0]

    def programa(self, filhos):
        return Programa(list(filhos))

    def bloco(self, filhos):
        return list(filhos[1:-1])

    def tipo_inteiro(self, filhos):
        return "inteiro"

    def tipo_decimal(self, filhos):
        return "decimal"

    def tipo_texto(self, filhos):
        return "texto"

    def tipo_logico(self, filhos):
        return "logico"

    def declaracao(self, filhos):
        tipo, nome = filhos[:2]
        expr = filhos[3] if len(filhos) == 5 else None
        return Decl(nome.line, tipo, str(nome), expr)

    def atribuicao(self, filhos):
        nome, _, expr, _ = filhos
        return Atrib(nome.line, str(nome), expr)

    def mostrar_stmt(self, filhos):
        palavra, expr, _ = filhos
        return Mostrar(palavra.line, expr)

    def se_stmt(self, filhos):
        palavra, cond, entao = filhos[:3]
        senao = filhos[4] if len(filhos) == 5 else None
        return Se(palavra.line, cond, entao, senao)

    def enquanto_stmt(self, filhos):
        palavra, cond, corpo = filhos
        return Enquanto(palavra.line, cond, corpo)

    def creditar_stmt(self, filhos):
        palavra, cliente, _, pontos, _ = filhos
        return Creditar(palavra.line, cliente, pontos)

    def resgatar_stmt(self, filhos):
        palavra, cupom, _, pontos, _ = filhos
        return Resgatar(palavra.line, cupom, pontos)

    def expirar_stmt(self, filhos):
        palavra, cliente, _, pontos, _ = filhos
        return Expirar(palavra.line, cliente, pontos)

    def extrato_stmt(self, filhos):
        palavra, cliente, _ = filhos
        return Extrato(palavra.line, cliente)

    def expr(self, filhos):
        return filhos[0]

    def logica_nao(self, filhos):
        return filhos[0]

    def _binaria(self, filhos):
        resultado = filhos[0]
        for indice in range(1, len(filhos), 2):
            operador, direita = filhos[indice], filhos[indice + 1]
            resultado = BinOp(operador.line, str(operador), resultado, direita)
        return resultado

    def logica_ou(self, filhos):
        return self._binaria(filhos)

    def logica_e(self, filhos):
        return self._binaria(filhos)

    def comparacao(self, filhos):
        return self._binaria(filhos)

    def soma_expr(self, filhos):
        return self._binaria(filhos)

    def produto_expr(self, filhos):
        return self._binaria(filhos)

    def unop(self, filhos):
        operador, expr = filhos
        return UnOp(operador.line, str(operador), expr)

    def fator(self, filhos):
        return filhos[0]

    def lit_decimal(self, filhos):
        token = filhos[0]
        return Lit(token.line, float(token), "decimal")

    def lit_inteiro(self, filhos):
        token = filhos[0]
        return Lit(token.line, int(token), "inteiro")

    def lit_texto(self, filhos):
        token = filhos[0]
        return Lit(token.line, str(token)[1:-1], "texto")

    def lit_logico(self, filhos):
        token = filhos[0]
        return Lit(token.line, token.type == "VERDADEIRO", "logico")

    def var(self, filhos):
        token = filhos[0]
        return Var(token.line, str(token))

    def agrupamento(self, filhos):
        return filhos[1]

def terminal_legivel(nome):
    nomes_amigaveis = {
        "NOME": "um identificador",
        "NUM_INTEIRO": "um número inteiro",
        "NUM_DECIMAL": "um número decimal",
        "TEXTO_LIT": "um texto entre aspas",
    }
    if nome in nomes_amigaveis:
        return nomes_amigaveis[nome]
    try:
        padrao = parser.get_terminal(nome).pattern
    except KeyError:
        return nome
    return repr(padrao.value) if padrao.type == "str" else nome

def analisar_sintaxe(fonte):
    tokenizar(fonte)
    try:
        arvore = parser.parse(fonte)
    except UnexpectedInput as erro:
        prioridade = {"PONTO_VIRGULA": 0, "VIRGULA": 1, "FECHA_CHAVE": 2, "FECHA_PAR": 3, "ATRIB": 4}
        terminais_esperados = set(getattr(erro, "expected", ()) or ())
        esperado = sorted(
            terminais_esperados,
            key=lambda nome: (prioridade.get(nome, 99), terminal_legivel(nome)),
        )
        lista_esperada = ", ".join(terminal_legivel(item) for item in esperado[:6])
        if not lista_esperada:
            lista_esperada = "uma continuação válida"
        token = getattr(erro, "token", None)
        fim_inesperado = token is None or getattr(token, "type", None) == "$END"
        linha = getattr(erro, "line", None) or fonte.count("\n") + 1
        coluna = getattr(erro, "column", None) or len(fonte.rsplit("\n", 1)[-1]) + 1
        mensagem = "fim inesperado do programa" if fim_inesperado else f"token inesperado {token.value!r}"
        dica = f"Esperado: {lista_esperada}"
        raise ErroCompilacao([Diagnostico("sintático", mensagem, linha, coluna, dica)]) from None
    return MontaAST().transform(arvore)

def mostrar_ast(no, nivel=0):
    prefixo = "  " * nivel
    if isinstance(no, list):
        print(f"{prefixo}[{len(no)} item(ns)]")
        for filho in no:
            mostrar_ast(filho, nivel + 1)
    elif is_dataclass(no):
        linha = f" (linha {no.linha})" if hasattr(no, "linha") else ""
        print(f"{prefixo}{type(no).__name__}{linha}")
        for campo in fields(no):
            if campo.name == "linha":
                continue
            print(f"{prefixo}  {campo.name}:")
            mostrar_ast(getattr(no, campo.name), nivel + 2)
    else:
        print(f"{prefixo}{no!r}")

PROGRAMA_SINTATICO = r'''
inteiro pontos = 100;
decimal compra = 250.0;
se compra >= 200.0 e pontos > 0 {
    creditar "cliente_42", pontos;
} senao {
    extrato "cliente_42";
}
enquanto pontos > 0 {
    expirar "cliente_42", 1;
    pontos = pontos - 1;
}
resgatar "cupom_10", 50;
'''

AST_FIDELIDADE = analisar_sintaxe(PROGRAMA_SINTATICO)
print("AST do programa FidelidadeScript:")
mostrar_ast(AST_FIDELIDADE)

EXEMPLO_PRECEDENCIA = "mostrar nao falso ou verdadeiro e 2 + 3 * 4 >= 10;"
print("\nAST para demonstrar precedência lógica, relacional e aritmética:")
mostrar_ast(analisar_sintaxe(EXEMPLO_PRECEDENCIA))

ERROS_SINTATICOS_DEMONSTRACAO = [
    "inteiro pontos = 50\nextrato \"cliente_42\";",
    "se verdadeiro { mostrar 1;",
]
for fonte_errada in ERROS_SINTATICOS_DEMONSTRACAO:
    try:
        analisar_sintaxe(fonte_errada)
    except ErroCompilacao as erro:
        print(f"\nExemplo de erro sintático:\n{erro}")
    else:
        raise AssertionError("O exemplo sintaticamente inválido deveria falhar.")

## Passo 3 — Análise semântica

A análise semântica verifica declarações, escopos de bloco, tipos e regras da FidelidadeScript. Erros impedem que o programa seja considerado válido; avisos informam situações sem bloquear a análise.

In [ ]:
# ============================================================
# PASSO 3 — Análise semântica da FidelidadeScript
# ============================================================

@dataclass
class Conv(Expr):
    expr: Expr
    tipo: str = "decimal"

class TabelaSimbolos:
    def __init__(self):
        self.escopos = [{}]

    def abrir(self):
        self.escopos.append({})

    def fechar(self):
        if len(self.escopos) == 1:
            raise RuntimeError("Não é possível fechar o escopo global.")
        return self.escopos.pop()

    def declarar(self, nome, tipo, linha, constante=None):
        atual = self.escopos[-1]
        if nome in atual:
            return False
        atual[nome] = {
            "tipo": tipo,
            "linha": linha,
            "usada": False,
            "constante": constante,
        }
        return True

    def buscar(self, nome):
        for escopo in reversed(self.escopos):
            if nome in escopo:
                return escopo[nome]
        return None

class AnalisadorSemantico:
    NUMERICOS = {"inteiro", "decimal"}
    RELACIONAIS = {">", "<", ">=", "<=", "==", "!="}

    def __init__(self):
        self.tabela = TabelaSimbolos()
        self.diagnosticos = []

    @property
    def erros(self):
        return [d for d in self.diagnosticos if d.nivel == "erro"]

    @property
    def avisos(self):
        return [d for d in self.diagnosticos if d.nivel == "aviso"]

    def erro(self, linha, mensagem, dica=""):
        self.diagnosticos.append(Diagnostico("semântico", mensagem, linha, 0, dica))

    def aviso(self, linha, mensagem, dica=""):
        self.diagnosticos.append(
            Diagnostico("semântico", mensagem, linha, 0, dica, nivel="aviso")
        )

    @staticmethod
    def eh_numero(valor):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool)

    def valor_constante(self, expr):
        if isinstance(expr, Lit):
            return expr.valor
        if isinstance(expr, Conv):
            valor = self.valor_constante(expr.expr)
            return None if valor is None else float(valor)
        if isinstance(expr, Var):
            info = self.tabela.buscar(expr.nome)
            return None if info is None else info["constante"]
        if isinstance(expr, UnOp):
            valor = self.valor_constante(expr.expr)
            if valor is None:
                return None
            if expr.op == "-" and self.eh_numero(valor):
                return -valor
            if expr.op == "+" and self.eh_numero(valor):
                return valor
            if expr.op == "nao" and isinstance(valor, bool):
                return not valor
            return None
        if isinstance(expr, BinOp):
            esquerda = self.valor_constante(expr.esq)
            direita = self.valor_constante(expr.dir)
            if esquerda is None or direita is None:
                return None
            if expr.op == "+":
                if self.eh_numero(esquerda) and self.eh_numero(direita):
                    return esquerda + direita
                if isinstance(esquerda, str) and isinstance(direita, str):
                    return esquerda + direita
                return None
            if expr.op in ("-", "*", "/"):
                if not self.eh_numero(esquerda) or not self.eh_numero(direita):
                    return None
                if expr.op == "-":
                    return esquerda - direita
                if expr.op == "*":
                    return esquerda * direita
                if direita != 0:
                    return esquerda / direita
                return None
            if expr.op == "e" and isinstance(esquerda, bool) and isinstance(direita, bool):
                return esquerda and direita
            if expr.op == "ou" and isinstance(esquerda, bool) and isinstance(direita, bool):
                return esquerda or direita
            if expr.op == "==":
                return esquerda == direita
            if expr.op == "!=":
                return esquerda != direita
            if self.eh_numero(esquerda) and self.eh_numero(direita):
                if expr.op == ">":
                    return esquerda > direita
                if expr.op == "<":
                    return esquerda < direita
                if expr.op == ">=":
                    return esquerda >= direita
                if expr.op == "<=":
                    return esquerda <= direita
        return None

    def converter(self, expr, tipo_destino, linha, contexto):
        if expr.tipo in (tipo_destino, None):
            return expr
        if expr.tipo == "inteiro" and tipo_destino == "decimal":
            return Conv(expr.linha, expr)
        dica = "Use decimal quando o valor tiver casas decimais." if (expr.tipo, tipo_destino) == ("decimal", "inteiro") else ""
        self.erro(linha, f"{contexto}: esperava {tipo_destino}, recebeu {expr.tipo}", dica)
        return expr

    def conferir_percentual(self, nome, expr, linha):
        if nome != "percentual_cashback":
            return
        if expr is None:
            return
        if expr.tipo is not None and expr.tipo not in self.NUMERICOS:
            self.erro(linha, "percentual de cashback precisa ser inteiro ou decimal")
            return
        valor = self.valor_constante(expr)
        if self.eh_numero(valor) and valor > 100:
            self.erro(linha, f"percentual de cashback inválido ({valor}%)", "o percentual máximo é 100%")
        elif self.eh_numero(valor) and valor == 0:
            self.aviso(linha, "percentual de cashback igual a 0%", "a campanha não concederá cashback")

    def analisar(self, programa):
        self.bloco(programa.comandos, novo_escopo=False)
        self.checar_nao_usadas(self.tabela.escopos[0])
        return programa

    def bloco(self, comandos, novo_escopo=True):
        if novo_escopo:
            self.tabela.abrir()
        for comando in comandos:
            self.comando(comando)
        if novo_escopo:
            self.checar_nao_usadas(self.tabela.fechar())

    def checar_nao_usadas(self, escopo):
        for nome, info in escopo.items():
            if not info["usada"]:
                self.aviso(info["linha"], f"'{nome}' foi declarada e nunca usada")

    def conferir_pontos(self, expr, linha, contexto):
        expr = self.converter(self.expr(expr), "inteiro", linha, contexto)
        valor = self.valor_constante(expr)
        if self.eh_numero(valor) and valor < 0:
            self.erro(linha, f"quantidade de pontos negativa ({valor})", "pontos não podem ser negativos")
        elif self.eh_numero(valor) and valor == 0 and contexto == "creditar":
            self.aviso(linha, "crédito de zero pontos", "a operação não altera o saldo do cliente")
        return expr

    def condicao(self, expr, linha, comando):
        expr = self.expr(expr)
        if expr.tipo not in ("logico", None):
            self.erro(linha, f"a condição de '{comando}' precisa ser logico, mas é {expr.tipo}")
        return expr

    def comando(self, comando):
        if isinstance(comando, Decl):
            if comando.expr is not None:
                comando.expr = self.converter(
                    self.expr(comando.expr), comando.tipo, comando.linha,
                    f"declaração de '{comando.nome}'",
                )
            constante = self.valor_constante(comando.expr) if comando.expr is not None else None
            if not self.tabela.declarar(comando.nome, comando.tipo, comando.linha, constante):
                self.erro(comando.linha, f"'{comando.nome}' já foi declarada neste bloco")
            self.conferir_percentual(comando.nome, comando.expr, comando.linha)
        elif isinstance(comando, Atrib):
            info = self.tabela.buscar(comando.nome)
            comando.expr = self.expr(comando.expr)
            if info is None:
                self.erro(comando.linha, f"'{comando.nome}' não foi declarada")
            else:
                comando.expr = self.converter(comando.expr, info["tipo"], comando.linha, f"atribuição a '{comando.nome}'")
                info["constante"] = self.valor_constante(comando.expr)
                self.conferir_percentual(comando.nome, comando.expr, comando.linha)
        elif isinstance(comando, Mostrar):
            comando.expr = self.expr(comando.expr)
        elif isinstance(comando, Se):
            comando.cond = self.condicao(comando.cond, comando.linha, "se")
            self.bloco(comando.entao)
            if comando.senao is not None:
                self.bloco(comando.senao)
        elif isinstance(comando, Enquanto):
            comando.cond = self.condicao(comando.cond, comando.linha, "enquanto")
            self.bloco(comando.corpo)
        elif isinstance(comando, Creditar):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "creditar (cliente)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "creditar")
        elif isinstance(comando, Resgatar):
            comando.cupom = self.converter(self.expr(comando.cupom), "texto", comando.linha, "resgatar (cupom)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "resgatar")
        elif isinstance(comando, Expirar):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "expirar (cliente)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "expirar")
        elif isinstance(comando, Extrato):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "extrato (cliente)")
        else:
            raise TypeError(f"Comando AST não suportado: {type(comando).__name__}")

    def expr(self, expr):
        if isinstance(expr, Lit):
            return expr
        if isinstance(expr, Conv):
            expr.expr = self.expr(expr.expr)
            return expr
        if isinstance(expr, Var):
            info = self.tabela.buscar(expr.nome)
            if info is None:
                self.erro(expr.linha, f"'{expr.nome}' não foi declarada", "declare a variável antes de usá-la")
            else:
                info["usada"] = True
                expr.tipo = info["tipo"]
            return expr
        if isinstance(expr, UnOp):
            expr.expr = self.expr(expr.expr)
            tipo = expr.expr.tipo
            if expr.op == "nao":
                if tipo not in ("logico", None):
                    self.erro(expr.linha, f"'nao' exige logico, recebeu {tipo}")
                expr.tipo = "logico"
            else:
                if tipo not in self.NUMERICOS | {None}:
                    self.erro(expr.linha, f"'{expr.op}' unário não aceita {tipo}")
                expr.tipo = tipo
            return expr
        if isinstance(expr, BinOp):
            expr.esq = self.expr(expr.esq)
            expr.dir = self.expr(expr.dir)
            tipo_esq, tipo_dir = expr.esq.tipo, expr.dir.tipo
            if tipo_esq is None or tipo_dir is None:
                expr.tipo = "logico" if expr.op in self.RELACIONAIS | {"e", "ou"} else tipo_esq or tipo_dir
                return expr
            if expr.op in ("e", "ou"):
                if tipo_esq != "logico" or tipo_dir != "logico":
                    self.erro(expr.linha, f"'{expr.op}' exige dois logico, recebeu {tipo_esq} e {tipo_dir}")
                expr.tipo = "logico"
            elif expr.op in self.RELACIONAIS:
                if tipo_esq in self.NUMERICOS and tipo_dir in self.NUMERICOS:
                    if "decimal" in (tipo_esq, tipo_dir):
                        expr.esq = self.converter(expr.esq, "decimal", expr.linha, "comparação")
                        expr.dir = self.converter(expr.dir, "decimal", expr.linha, "comparação")
                elif expr.op in ("==", "!=") and tipo_esq == tipo_dir:
                    pass
                else:
                    self.erro(expr.linha, f"comparação '{expr.op}' inválida entre {tipo_esq} e {tipo_dir}")
                expr.tipo = "logico"
            elif expr.op == "+" and tipo_esq == tipo_dir == "texto":
                expr.tipo = "texto"
            elif expr.op in ("+", "-", "*", "/") and tipo_esq in self.NUMERICOS and tipo_dir in self.NUMERICOS:
                if expr.op == "/" and self.valor_constante(expr.dir) == 0:
                    self.erro(expr.linha, "divisão por zero")
                if expr.op == "/" or "decimal" in (tipo_esq, tipo_dir):
                    expr.esq = self.converter(expr.esq, "decimal", expr.linha, "operação aritmética")
                    expr.dir = self.converter(expr.dir, "decimal", expr.linha, "operação aritmética")
                    expr.tipo = "decimal"
                else:
                    expr.tipo = "inteiro"
            else:
                self.erro(expr.linha, f"operação '{expr.op}' inválida entre {tipo_esq} e {tipo_dir}")
                expr.tipo = tipo_esq
            return expr
        raise TypeError(f"Expressão AST não suportada: {type(expr).__name__}")

def analisar_semantica(fonte):
    programa = analisar_sintaxe(fonte)
    analisador = AnalisadorSemantico()
    analisador.analisar(programa)
    return programa, analisador

GALERIA_SEMANTICA = [
    ("variável não declarada", 'mostrar saldo_inexistente;'),
    ("tipo incompatível", "inteiro pontos = 3.5;"),
    ("declaração duplicada", "inteiro pontos = 10; inteiro pontos = 20;"),
    ("condição não lógica", "se 5 { mostrar 1; }"),
    ("operação entre texto e número", 'mostrar "cliente_42" - 2;'),
    ("variável fora do escopo", "se verdadeiro { inteiro bonus = 10; mostrar bonus; } mostrar bonus;"),
    ("cashback acima de 100%", "decimal percentual_cashback = 150.0; mostrar percentual_cashback;"),
    ("pontos negativos", 'creditar "cliente_42", -5;'),
]

for titulo, fonte in GALERIA_SEMANTICA:
    programa = analisar_sintaxe(fonte)
    analisador = AnalisadorSemantico()
    analisador.analisar(programa)
    if not analisador.erros:
        raise AssertionError(f"O caso '{titulo}' deveria produzir erro semântico.")
    print(f"\n--- {titulo} ---")
    for diagnostico in analisador.diagnosticos:
        print(diagnostico)

PROGRAMA_COERCAO = "decimal cashback = 10; mostrar cashback;"
ast_coercao, sem_coercao = analisar_semantica(PROGRAMA_COERCAO)
if sem_coercao.erros or not isinstance(ast_coercao.comandos[0].expr, Conv):
    raise AssertionError("A coerção inteiro para decimal não foi aplicada corretamente.")
print("\nCoerção inteiro -> decimal inserida na AST:")
mostrar_ast(ast_coercao.comandos[0])

PROGRAMA_AVISOS = 'decimal percentual_cashback = 0.0; creditar "cliente_42", 0;'
_, sem_avisos = analisar_semantica(PROGRAMA_AVISOS)
if sem_avisos.erros or not sem_avisos.avisos:
    raise AssertionError("O programa de demonstração deveria produzir avisos sem erros.")
print("\nExemplo de avisos (não bloqueiam a análise):")
for diagnostico in sem_avisos.avisos:
    print(diagnostico)
